In [0]:
# 1. Configuración

import json
import time
from datetime import datetime
import pandas as pd

CATALOG = "workspace"
SCHEMA  = "si7006_t2"
VOL_RAW       = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data"
VOL_STREAM_IN = f"/Volumes/{CATALOG}/{SCHEMA}/stream_input"
CSV_ORIGEN    = f"{VOL_RAW}/OnlineRetail.csv"

EVENTOS_POR_LOTE     = 50
SEGUNDOS_ENTRE_LOTES = 5
LOTES_TOTALES        = 30

print(f"Origen:           {CSV_ORIGEN}")
print(f"Destino:          {VOL_STREAM_IN}")
print(f"Eventos/lote:     {EVENTOS_POR_LOTE}")
print(f"Lotes totales:    {LOTES_TOTALES}")
print(f"Eventos totales:  {LOTES_TOTALES * EVENTOS_POR_LOTE:,}")

Origen:           /Volumes/workspace/si7006_t2/raw_data/OnlineRetail.csv
Destino:          /Volumes/workspace/si7006_t2/stream_input
Eventos/lote:     50
Lotes totales:    30
Eventos totales:  1,500


In [0]:
# 2. Cargar el CSV en memoria y mezclar

df_pandas = pd.read_csv(
    CSV_ORIGEN,
    encoding="ISO-8859-1",
    dtype={
        "InvoiceNo": str, "StockCode": str, "Description": str,
        "Quantity": "Int64", "UnitPrice": float,
        "CustomerID": "Int64", "Country": str
    },
    parse_dates=["InvoiceDate"]
)

df_pandas = df_pandas.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Filas cargadas: {len(df_pandas):,}")
print(df_pandas.head(3))

Filas cargadas: 541,909
  InvoiceNo StockCode  ... CustomerID         Country
0    555200     71459  ...      17315  United Kingdom
1    554974     21128  ...      14031  United Kingdom
2    550972     21086  ...      14031  United Kingdom

[3 rows x 8 columns]


In [0]:
# 3. Limpiar el directorio stream_input antes de empezar

for archivo in dbutils.fs.ls(VOL_STREAM_IN):
    dbutils.fs.rm(archivo.path)

print(f"Directorio limpio: {len(dbutils.fs.ls(VOL_STREAM_IN))} archivos")

✓ Archivo de prueba escrito: /Volumes/workspace/si7006_t2/stream_input/test_inicial_20260428_163003.json

Contenido (primeras 500 chars):
{"InvoiceNo":"555200","StockCode":"71459","Description":"HANGING JAM JAR T-LIGHT HOLDER","Quantity":24,"InvoiceDate":"2011-06-01T12:05:00.000","UnitPrice":0.85,"CustomerID":17315,"Country":"United Kingdom"}
{"InvoiceNo":"554974","StockCode":"21128","Description":"GOLD FISHING GNOME","Quantity":4,"InvoiceDate":"2011-05-27T17:14:00.000","UnitPrice":6.95,"CustomerID":14031,"Country":"United Kingdom"}
{"InvoiceNo":"550972","StockCode":"21086","Description":"SET\/6 RED SPOTTY PAPER CUPS","Quantity":4


In [0]:
# 4. Loop de generación de lotes

filas_totales = len(df_pandas)
posicion = 0

for lote in range(1, LOTES_TOTALES + 1):
    inicio = posicion % filas_totales
    fin = inicio + EVENTOS_POR_LOTE
    df_lote = df_pandas.iloc[inicio:fin].copy()

    # Reescribir InvoiceDate con timestamp actual para simular ventas en vivo
    df_lote["InvoiceDate"] = datetime.now()

    nombre = f"orders_{datetime.now().strftime('%Y%m%d_%H%M%S')}_lote{lote:04d}.json"
    ruta = f"{VOL_STREAM_IN}/{nombre}"

    with open(ruta, "w", encoding="utf-8") as f:
        f.write(df_lote.to_json(orient="records", lines=True, date_format="iso"))

    print(f"[{lote:03d}/{LOTES_TOTALES}] {datetime.now().strftime('%H:%M:%S')} -> {nombre}")

    posicion += EVENTOS_POR_LOTE
    if lote < LOTES_TOTALES:
        time.sleep(SEGUNDOS_ENTRE_LOTES)

print(f"\nTerminado. {LOTES_TOTALES * EVENTOS_POR_LOTE:,} eventos generados.")

[001/30] 16:42:35 -> orders_20260429_164234_lote0001.json
[002/30] 16:42:40 -> orders_20260429_164240_lote0002.json
[003/30] 16:42:45 -> orders_20260429_164245_lote0003.json
[004/30] 16:42:50 -> orders_20260429_164250_lote0004.json
[005/30] 16:42:55 -> orders_20260429_164255_lote0005.json
[006/30] 16:43:01 -> orders_20260429_164300_lote0006.json
[007/30] 16:43:06 -> orders_20260429_164306_lote0007.json
[008/30] 16:43:11 -> orders_20260429_164311_lote0008.json
[009/30] 16:43:16 -> orders_20260429_164316_lote0009.json
[010/30] 16:43:21 -> orders_20260429_164321_lote0010.json
[011/30] 16:43:26 -> orders_20260429_164326_lote0011.json
[012/30] 16:43:31 -> orders_20260429_164331_lote0012.json
[013/30] 16:43:36 -> orders_20260429_164336_lote0013.json
[014/30] 16:43:42 -> orders_20260429_164341_lote0014.json
[015/30] 16:43:47 -> orders_20260429_164347_lote0015.json
[016/30] 16:43:52 -> orders_20260429_164352_lote0016.json
[017/30] 16:43:57 -> orders_20260429_164357_lote0017.json
[018/30] 16:44